In [8]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Click Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")

GPU available: True
GPU name: Tesla T4


In [9]:
import zipfile, os

ZIP_PATH = '/content/nexus_gaze_data.zip'
print("Zip found:", os.path.exists(ZIP_PATH))

with zipfile.ZipFile(ZIP_PATH) as z:
    for m in z.infolist():
        name = m.filename.replace('\\', '/')     # fix Windows-style paths
        if name.endswith('/'):
            continue
        target = os.path.join('/content', name)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(m) as src, open(target, 'wb') as dst:
            dst.write(src.read())

n_images = len(os.listdir('/content/gaze_random/images'))
with open('/content/gaze_random/labels.csv') as f:
    n_rows = sum(1 for _ in f) - 1
print("Images:", n_images, "| Label rows:", n_rows)

Zip found: True
Images: 17186 | Label rows: 17186


In [10]:
import cv2
import numpy as np
import pandas as pd

BASE = '/content/gaze_random/'
df = pd.read_csv(BASE + 'labels.csv')

# ---- Split by SESSION, so the test dots are never seen in training ----
VAL_SESSION = 'tg20261109_103153'                           # glasses, for tuning
TEST_SESSIONS = ['t20261109_094441', 'tg20261109_103411']   # final test (no glasses, glasses)

def split_of(session):
    if session == VAL_SESSION:
        return 'val'
    if session in TEST_SESSIONS:
        return 'test'
    return 'train'

df['split'] = df['session'].apply(split_of)
df['glasses'] = df['session'].str[1] == 'g'

# ---- Remove blink frames (using the saved eye-openness values) ----
# A frame is dropped if the eyes are much more closed than usual for that dot.
df['min_ear'] = df[['left_ear', 'right_ear']].min(axis=1)
median_ear = df.groupby(['session', 'point'])['min_ear'].transform('median')
keep = (df['min_ear'] >= 0.6 * median_ear) & (df['min_ear'] > 0.07)
print("Blink-like frames removed:", int((~keep).sum()), "of", len(df))
df = df[keep].reset_index(drop=True)

# ---- Load the images and labels ----
X = np.stack([cv2.imread(BASE + p, cv2.IMREAD_GRAYSCALE) for p in df['file']])
y = df[['target_x', 'target_y']].values.astype('float32')
lm_cols = [c for c in df.columns if c.startswith('lm')]
F = df[lm_cols].values.astype('float32')

print("Images array:", X.shape, "| Labels:", y.shape, "| Landmark features:", F.shape)
print()
print(df.groupby(['split', 'glasses']).size())

Blink-like frames removed: 1718 of 17186
Images array: (15468, 48, 128) | Labels: (15468, 2) | Landmark features: (15468, 22)

split  glasses
test   False      1841
       True       2061
train  False      5500
       True       4042
val    True       2024
dtype: int64


In [11]:
import torch
import torch.nn as nn
import numpy as np

SCREEN_W, SCREEN_H = 1920, 1080    # change these if your screen has a different size
torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Masks for the three sets (rows of df) ----
tr, va, te = [(df['split'] == s).values for s in ('train', 'val', 'test')]
splits = {'train': tr, 'val': va, 'test': te}

# ---- Normalise the landmark numbers using TRAIN data only ----
F_mean, F_std = F[tr].mean(axis=0), F[tr].std(axis=0) + 1e-6
Fn = (F - F_mean) / F_std

def to_t(a):
    return torch.tensor(a, dtype=torch.float32, device=device)

Ft = {k: to_t(Fn[m]) for k, m in splits.items()}   # landmark numbers
yt = {k: to_t(y[m]) for k, m in splits.items()}    # targets (x, y as fractions of the screen)

# ---- Metrics ----
scale = torch.tensor([SCREEN_W, SCREEN_H], dtype=torch.float32, device=device)

def pixel_error(pred, target):
    """Distance between predicted and true screen position, in pixels."""
    return ((pred - target) * scale).norm(dim=1)

def zone9(p):
    """Which of the 3 x 3 screen zones a position falls in (0 to 8)."""
    col = (p[:, 0] * 3).clamp(0, 2.999).long()
    row = (p[:, 1] * 3).clamp(0, 2.999).long()
    return row * 3 + col

def report(title, predict, split='test'):
    with torch.no_grad():
        p = predict(split)
    t = yt[split]
    err = pixel_error(p, t).cpu().numpy()
    acc = (zone9(p) == zone9(t)).float().cpu().numpy()
    if split == 'test':
        g = df.loc[te, 'glasses'].values
        groups = [('all', np.ones(len(g), bool)), ('no glasses', ~g), ('glasses', g)]
    else:
        groups = [('all', np.ones(len(err), bool))]
    print(f"{title} on {split}")
    for name, m in groups:
        print(f"  {name:11s} mean {err[m].mean():6.0f} px | median {np.median(err[m]):6.0f} px"
              f" | 9-zone accuracy {acc[m].mean() * 100:5.1f}%  (n={m.sum()})")

# ---- Baseline model: landmark numbers only ----
mlp = nn.Sequential(
    nn.Linear(22, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 2),
).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.SmoothL1Loss()

best_err, best_state = 1e9, None
for epoch in range(60):
    mlp.train()
    order = torch.randperm(len(Ft['train']), device=device)
    for i in range(0, len(order), 256):
        b = order[i:i + 256]
        optimizer.zero_grad()
        loss = loss_fn(mlp(Ft['train'][b]), yt['train'][b])
        loss.backward()
        optimizer.step()
    mlp.eval()
    with torch.no_grad():
        val_err = pixel_error(mlp(Ft['val']), yt['val']).mean().item()
    if val_err < best_err:
        best_err = val_err
        best_state = {k: v.clone() for k, v in mlp.state_dict().items()}
    if epoch % 10 == 9:
        print(f"epoch {epoch + 1:2d} | validation error {val_err:6.0f} px")

mlp.load_state_dict(best_state)
mlp.eval()
print()
report("BASELINE (landmarks only)", lambda s: mlp(Ft[s]), 'val')
report("BASELINE (landmarks only)", lambda s: mlp(Ft[s]), 'test')

epoch 10 | validation error    474 px
epoch 20 | validation error    521 px
epoch 30 | validation error    468 px
epoch 40 | validation error    542 px
epoch 50 | validation error    491 px
epoch 60 | validation error    526 px

BASELINE (landmarks only) on val
  all         mean    453 px | median    426 px | 9-zone accuracy  28.2%  (n=2024)
BASELINE (landmarks only) on test
  all         mean    522 px | median    491 px | 9-zone accuracy  18.8%  (n=3902)
  no glasses  mean    634 px | median    671 px | 9-zone accuracy  20.7%  (n=1841)
  glasses     mean    422 px | median    419 px | 9-zone accuracy  17.0%  (n=2061)


In [12]:
# ---- Build gaze-friendly features from the saved landmarks ----
# Landmark order saved by the collection script: nose, forehead, chin,
# face side A, face side B, eye A outer, eye A inner, eye B inner,
# eye B outer, iris A, iris B. Each has an x and a y.
P = F.reshape(len(F), 11, 2) * np.array([640.0, 480.0], dtype=np.float32)  # to pixels
nose, fore, chin, sideA, sideB = P[:, 0], P[:, 1], P[:, 2], P[:, 3], P[:, 4]
eA_out, eA_in, eB_in, eB_out, irisA, irisB = P[:, 5], P[:, 6], P[:, 7], P[:, 8], P[:, 9], P[:, 10]

def iris_offset(c1, c2, iris):
    """Where the iris sits inside the eye: t = along the eye, s = across it (in eye widths)."""
    axis = c2 - c1
    L = np.linalg.norm(axis, axis=1) + 1e-6
    u = axis / L[:, None]
    n = np.stack([-u[:, 1], u[:, 0]], axis=1)
    rel = iris - (c1 + c2) / 2
    return (rel * u).sum(1) / L, (rel * n).sum(1) / L, L

tA, sA, LA = iris_offset(eA_out, eA_in, irisA)
tB, sB, LB = iris_offset(eB_in, eB_out, irisB)

face_w = np.linalg.norm(sideB - sideA, axis=1) + 1e-6
face_h = np.linalg.norm(chin - fore, axis=1) + 1e-6
mid_sides = (sideA + sideB) / 2
mid_vert = (fore + chin) / 2
yaw = (nose[:, 0] - mid_sides[:, 0]) / face_w
pitch = (nose[:, 1] - mid_vert[:, 1]) / face_h
roll = np.arctan2(sideB[:, 1] - sideA[:, 1], sideB[:, 0] - sideA[:, 0])
cx, cy = mid_sides[:, 0] / 640.0, mid_vert[:, 1] / 480.0

G = np.stack([tA, sA, tB, sB, yaw, pitch, roll, cx, cy,
              face_w / 640.0, LA / face_w, LB / face_w], axis=1).astype('float32')
G_mean, G_std = G[tr].mean(0), G[tr].std(0) + 1e-6      # statistics from TRAIN only
Gn = (G - G_mean) / G_std
Gt = {k: to_t(Gn[m]) for k, m in splits.items()}

# ---- A reusable training function for small networks on feature numbers ----
def train_mlp(feats, in_dim, epochs=60, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 2),
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss()
    best, best_state = 1e9, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(feats['train']), device=device)
        for i in range(0, len(order), 256):
            b = order[i:i + 256]
            opt.zero_grad()
            loss_fn(model(feats['train'][b]), yt['train'][b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = pixel_error(model(feats['val']), yt['val']).mean().item()
        if v < best:
            best = v
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        if epoch % 20 == 19:
            print(f"epoch {epoch + 1} | validation error {v:.0f} px")
    model.load_state_dict(best_state)
    model.eval()
    return model

mlp2 = train_mlp(Gt, G.shape[1])

print()
center = torch.tensor([0.5, 0.5], device=device)
report("GUESS THE CENTER (reference)", lambda s: center.expand(len(yt[s]), 2), 'test')
print()
report("IMPROVED BASELINE (gaze features)", lambda s: mlp2(Gt[s]), 'val')
report("IMPROVED BASELINE (gaze features)", lambda s: mlp2(Gt[s]), 'test')

epoch 20 | validation error 280 px
epoch 40 | validation error 296 px
epoch 60 | validation error 302 px

GUESS THE CENTER (reference) on test
  all         mean    541 px | median    524 px | 9-zone accuracy  11.7%  (n=3902)
  no glasses  mean    530 px | median    515 px | 9-zone accuracy  12.1%  (n=1841)
  glasses     mean    551 px | median    530 px | 9-zone accuracy  11.4%  (n=2061)

IMPROVED BASELINE (gaze features) on val
  all         mean    258 px | median    234 px | 9-zone accuracy  47.3%  (n=2024)
IMPROVED BASELINE (gaze features) on test
  all         mean    310 px | median    262 px | 9-zone accuracy  38.9%  (n=3902)
  no glasses  mean    423 px | median    420 px | 9-zone accuracy  20.7%  (n=1841)
  glasses     mean    210 px | median    193 px | 9-zone accuracy  55.2%  (n=2061)


In [13]:
import time
import torch.nn.functional as Fn_

# ---- Images as GPU tensors (uint8 to save memory) ----
Xt = {k: torch.tensor(X[m], dtype=torch.uint8, device=device).unsqueeze(1)
      for k, m in splits.items()}                      # shape (N, 1, 48, 128)

def prep(xb):
    return (xb.float() / 255.0 - 0.5) / 0.5            # scale pixels to about -1..1

def augment(img):
    """Training only: random brightness/contrast, a little noise, a tiny shift."""
    c = 1.0 + 0.3 * (torch.rand(len(img), 1, 1, 1, device=device) * 2 - 1)
    b = 0.2 * (torch.rand(len(img), 1, 1, 1, device=device) * 2 - 1)
    img = img * c + b + 0.05 * torch.randn_like(img)
    dx, dy = np.random.randint(-2, 3, size=2)
    return torch.roll(img, shifts=(int(dy), int(dx)), dims=(2, 3))

# ---- The model: CNN on the eye picture + small network on the gaze features ----
class GazeNet(nn.Module):
    def __init__(self, n_feats):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1),
                                 nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
        self.cnn = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 64))
        self.fc_img = nn.Sequential(nn.Flatten(), nn.Dropout(0.3),
                                    nn.Linear(64 * 3 * 8, 128), nn.ReLU())
        self.fc_feat = nn.Sequential(nn.Linear(n_feats, 64), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(128 + 64, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, 2))

    def forward(self, img, feats):
        return self.head(torch.cat([self.fc_img(self.cnn(img)), self.fc_feat(feats)], dim=1))

def predict_cnn(model, split, bs=512):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(yt[split]), bs):
            outs.append(model(prep(Xt[split][i:i + bs]), Gt[split][i:i + bs]))
    return torch.cat(outs)

def train_cnn(epochs=40, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = GazeNet(G.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.L1Loss()
    best, best_state = 1e9, None
    n = len(yt['train'])
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        order = torch.randperm(n, device=device)
        total = 0.0
        for i in range(0, n, 128):
            b = order[i:i + 128]
            opt.zero_grad()
            loss = loss_fn(model(augment(prep(Xt['train'][b])), Gt['train'][b]), yt['train'][b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        sched.step()
        val_err = pixel_error(predict_cnn(model, 'val'), yt['val']).mean().item()
        if val_err < best:
            best = val_err
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"epoch {epoch + 1:2d} | train loss {total / n:.4f} | "
              f"validation error {val_err:5.0f} px | {time.time() - t0:.1f}s")
    model.load_state_dict(best_state)
    model.eval()
    return model

cnn = train_cnn()

print()
report("GAZE CNN (eye image + gaze features)", lambda s: predict_cnn(cnn, s), 'val')
report("GAZE CNN (eye image + gaze features)", lambda s: predict_cnn(cnn, s), 'test')

# ---- Save the model and the numbers the live program will need ----
torch.save({'model': cnn.state_dict(), 'G_mean': G_mean, 'G_std': G_std,
            'screen': (SCREEN_W, SCREEN_H)}, '/content/gaze_model.pt')
from google.colab import files
files.download('/content/gaze_model.pt')

epoch  1 | train loss 0.1372 | validation error   296 px | 2.9s
epoch  2 | train loss 0.0905 | validation error   262 px | 1.6s
epoch  3 | train loss 0.0837 | validation error   316 px | 1.6s
epoch  4 | train loss 0.0781 | validation error   262 px | 1.6s
epoch  5 | train loss 0.0774 | validation error   286 px | 1.9s
epoch  6 | train loss 0.0725 | validation error   275 px | 1.4s
epoch  7 | train loss 0.0710 | validation error   270 px | 1.4s
epoch  8 | train loss 0.0686 | validation error   265 px | 1.4s
epoch  9 | train loss 0.0681 | validation error   239 px | 1.4s
epoch 10 | train loss 0.0652 | validation error   230 px | 1.4s
epoch 11 | train loss 0.0653 | validation error   246 px | 1.4s
epoch 12 | train loss 0.0644 | validation error   243 px | 1.4s
epoch 13 | train loss 0.0615 | validation error   225 px | 1.4s
epoch 14 | train loss 0.0622 | validation error   231 px | 1.4s
epoch 15 | train loss 0.0610 | validation error   251 px | 1.4s
epoch 16 | train loss 0.0597 | validatio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# ---- Experiment: does a short calibration at the start fix per-session errors? ----
with torch.no_grad():
    pred_test = predict_cnn(cnn, 'test').cpu().numpy()
true_test = yt['test'].cpu().numpy()
meta = df.loc[te, ['session', 'point']].reset_index(drop=True)
rng = np.random.default_rng(0)
scale_np = np.array([SCREEN_W, SCREEN_H])

def err_px(p, t):
    return np.linalg.norm((p - t) * scale_np, axis=1)

def calibrated_error(session, k, repeats=30):
    """Fit a small correction on k dots, then measure the error on the other dots."""
    idx = np.where(meta['session'].values == session)[0]
    points = np.unique(meta['point'].values[idx])
    before, after = [], []
    for _ in range(repeats):
        cal_pts = rng.choice(points, size=k, replace=False)
        is_cal = np.isin(meta['point'].values[idx], cal_pts)
        cal_i, ev_i = idx[is_cal], idx[~is_cal]
        A = np.c_[pred_test[cal_i], np.ones(len(cal_i))]
        W, *_ = np.linalg.lstsq(A, true_test[cal_i], rcond=None)
        fixed = np.c_[pred_test[ev_i], np.ones(len(ev_i))] @ W
        before.append(err_px(pred_test[ev_i], true_test[ev_i]).mean())
        after.append(err_px(fixed, true_test[ev_i]).mean())
    return np.mean(before), np.mean(after)

names = {'t20261109_094441': 'early test run',
         'tg20261109_103411': 'glasses test run'}
print("Mean error on dots NOT used for calibration:")
for sess, label in names.items():
    for k in (9, 15, 30):
        b, a = calibrated_error(sess, k)
        print(f"  {label:17s} | {k:2d} calibration dots | "
              f"no calibration {b:4.0f} px -> with calibration {a:4.0f} px")

Mean error on dots NOT used for calibration:
  early test run    |  9 calibration dots | no calibration  349 px -> with calibration  202 px
  early test run    | 15 calibration dots | no calibration  348 px -> with calibration  188 px
  early test run    | 30 calibration dots | no calibration  346 px -> with calibration  183 px
  glasses test run  |  9 calibration dots | no calibration  203 px -> with calibration  159 px
  glasses test run  | 15 calibration dots | no calibration  206 px -> with calibration  144 px
  glasses test run  | 30 calibration dots | no calibration  208 px -> with calibration  139 px
